In [1]:
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, Subset, DataLoader
import numpy as np

from dataset import create_dataloaders
from tcn_model import MEGTCN
from gan_model_simple import MEGGAN
from cnn_baseline_1d import CNNBaseline1D
from train import train_one_epoch
from evaluate import evaluate, evaluate_top_models_person_cv
from grid_search import run_grid_search, run_person_grid_search

In [2]:
BATCH_SIZE = 8

In [3]:
INTRA_DATA_DIR = "preprocessed_data/Intra"
intra_train_loader, intra_test_loader = create_dataloaders(INTRA_DATA_DIR, BATCH_SIZE, add_person_id=True)

Loading test data...
Loading train data...
Loaded 32 training samples
Loaded 8 test samples
Class distribution in training: [8 8 8 8]
Class distribution in test: [2 2 2 2]
Train batches: 4
Test batches: 1


### Cross Dataset

In [4]:
CROSS_DATA_DIR = "preprocessed_data/Cross"
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-3
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

cross_train_loader, cross_test_loader = create_dataloaders(CROSS_DATA_DIR, BATCH_SIZE, add_person_id=True)

Using device: cuda
Loading test1 data...
Loading test2 data...
Loading test3 data...
Loading train data...
Loaded 64 training samples
Loaded 48 test samples
Class distribution in training: [16 16 16 16]
Class distribution in test: [12 12 12 12]
Train batches: 8
Test batches: 6


In [5]:
cross_dataset = cross_train_loader.dataset
intra_dataset = intra_train_loader.dataset

def build_person_subsets(dataset, source_name):
    """Build subsets of the dataset grouped by person_id."""
    if dataset.person_ids is None:
        raise ValueError(f"{source_name} dataset must be created with add_person_id=True")

    grouped_indices = {}
    for index, person_id in enumerate(dataset.person_ids):
        grouped_indices.setdefault(int(person_id), []).append(index)

    return [
        (f"{source_name}_{person_id}", Subset(dataset, indices), person_id)
        for person_id, indices in sorted(grouped_indices.items())
    ]


cross_person_splits = build_person_subsets(cross_dataset, "cross")
intra_person_splits = build_person_subsets(intra_dataset, "intra")
person_splits = cross_person_splits + intra_person_splits

print("Person-level folds:")
for split_name, subset, person_id in person_splits:
    print(f"  {split_name}: person_id={person_id}, samples={len(subset)}")

Person-level folds:
  cross_113922: person_id=113922, samples=32
  cross_164636: person_id=164636, samples=32
  intra_105923: person_id=105923, samples=32


In [6]:
def cross_validation(
    model_class,
    person_splits,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    model_kwargs=None,
    weight_decay=1e-4,
    patience=8,
    min_delta=1e-3,
):
    """Train one model per person split with early stopping.

    The key anti-overfitting pieces here are:
    - weight decay in the optimizer
    - learning-rate reduction when validation loss plateaus
    - early stopping that restores the best validation checkpoint
    """

    fold_accuracies = []
    model_kwargs = model_kwargs or {}
    use_pin_memory = DEVICE == "cuda"

    for fold, (test_name, test_subset, test_person_id) in enumerate(person_splits):
        train_subsets = [
            subset
            for split_name, subset, _ in person_splits
            if split_name != test_name
        ]

        fold_train_dataset = ConcatDataset(train_subsets)
        fold_train_loader = DataLoader(
            fold_train_dataset,
            batch_size=batch_size,
            shuffle=True,
            pin_memory=use_pin_memory,
        )
        val_loader = DataLoader(
            test_subset,
            batch_size=batch_size,
            shuffle=False,
            pin_memory=use_pin_memory,
        )

        model = model_class(num_classes=NUM_CLASSES, **model_kwargs).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            patience=max(1, patience // 2),
            factor=0.5,
        )

        print(
            f"Starting fold {fold + 1}/{len(person_splits)} | "
            f"Test person: {test_name} (person_id={test_person_id})"
        )

        best_val_loss = float("inf")
        best_val_acc = 0.0
        best_state_dict = None
        bad_epochs = 0

        for epoch in range(epochs):
            train_loss, train_acc = train_one_epoch(
                model,
                fold_train_loader,
                criterion,
                optimizer,
                DEVICE,
            )
            val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
            scheduler.step(val_loss)

            # Keep the checkpoint with the lowest validation loss.
            if val_loss < best_val_loss - min_delta:
                best_val_loss = val_loss
                best_val_acc = val_acc
                best_state_dict = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }
                bad_epochs = 0
            else:
                bad_epochs += 1

            print(
                f"Fold {fold + 1} | Epoch {epoch + 1}/{epochs} | "
                f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | "
                f"Val Loss: {val_loss:.4f}"
            )

            # Stop once validation loss has not improved for several epochs.
            if bad_epochs >= patience:
                print(
                    f"Early stopping at epoch {epoch + 1} after {patience} "
                    "non-improving epochs."
                )
                break

        # Restore the best model before recording the fold result.
        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)

        fold_accuracies.append(best_val_acc)

    print(
        f"Mean person-level validation accuracy: {np.mean(fold_accuracies):.2f}%"
    )

# Grid Search, using person splits

In [7]:
cross_person_splits = build_person_subsets(cross_dataset, "cross")
intra_person_splits = build_person_subsets(intra_dataset, "intra")
person_splits = cross_person_splits + intra_person_splits

print("Person-level folds:")
for split_name, subset, person_id in person_splits:
    print(f"  {split_name}: person_id={person_id}, samples={len(subset)}")

Person-level folds:
  cross_113922: person_id=113922, samples=32
  cross_164636: person_id=164636, samples=32
  intra_105923: person_id=105923, samples=32


In [11]:
# from gan_model_simple import MEGGAN

param_grid = {
    "learning_rate": [5e-3],
    "temporal_hidden": [64],
    "graph_hidden": [16, 64],
    "dropout": [0.1],
    "batch_size": [8],
}

top_models = run_person_grid_search(
    model_class=MEGGAN,
    param_grid=param_grid,
    person_splits=person_splits,
    num_classes=4,
    epochs=100,
    patience=20,
)

Total configs: 2

Testing parameters:
{'learning_rate': 0.005, 'temporal_hidden': 64, 'graph_hidden': 16, 'dropout': 0.1, 'batch_size': 8}

Fold 1/3 | Test person: 113922
Fold 1 | Epoch 1/100 | Train Acc: 25.00% | Val Acc: 25.00% | Val Loss: 1.3706
Fold 1 | Epoch 2/100 | Train Acc: 34.38% | Val Acc: 25.00% | Val Loss: 1.3613
Fold 1 | Epoch 3/100 | Train Acc: 50.00% | Val Acc: 37.50% | Val Loss: 1.3160
Fold 1 | Epoch 4/100 | Train Acc: 46.88% | Val Acc: 37.50% | Val Loss: 1.3277
Fold 1 | Epoch 5/100 | Train Acc: 37.50% | Val Acc: 46.88% | Val Loss: 1.2318
Fold 1 | Epoch 6/100 | Train Acc: 56.25% | Val Acc: 46.88% | Val Loss: 1.1660
Fold 1 | Epoch 7/100 | Train Acc: 51.56% | Val Acc: 37.50% | Val Loss: 1.1471
Fold 1 | Epoch 8/100 | Train Acc: 54.69% | Val Acc: 59.38% | Val Loss: 1.0982
Fold 1 | Epoch 9/100 | Train Acc: 59.38% | Val Acc: 31.25% | Val Loss: 1.1953
Fold 1 | Epoch 10/100 | Train Acc: 64.06% | Val Acc: 59.38% | Val Loss: 1.0348
Fold 1 | Epoch 11/100 | Train Acc: 62.50% | Val 